# 02. 에이전트 학습 — PPO로 CartPole 풀기

**Stable-Baselines3(SB3)** 는 검증된 강화학습 알고리즘 구현을 제공하는 라이브러리입니다.
대표 알고리즘 **PPO(Proximal Policy Optimization)** 로 CartPole을 학습시켜,
무작위 정책 대비 성능이 얼마나 향상되는지 확인합니다.

1. 학습 전 성능 측정
2. PPO 학습
3. 학습 후 성능 측정 및 비교

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

import torch
print('GPU:', torch.cuda.is_available())

## 1. 학습 전 성능

PPO 에이전트를 만들고, **학습하기 전** 성능을 먼저 측정합니다. (정책 가중치가 무작위 초기값이라 낮게 나옵니다.)
`evaluate_policy`는 여러 에피소드의 평균 보상과 표준편차를 돌려줍니다.

In [ ]:
env = gym.make('CartPole-v1')
model = PPO('MlpPolicy', env, verbose=0)

mean_before, std_before = evaluate_policy(model, env, n_eval_episodes=20)
print(f'학습 전 평균 보상: {mean_before:.1f} +/- {std_before:.1f}')

## 2. PPO 학습

`learn`으로 학습합니다. CartPole은 가벼워 수만 스텝이면 충분히 풉니다.
(`total_timesteps`를 늘리면 더 안정적으로 최고 점수에 도달합니다.)

In [ ]:
model.learn(total_timesteps=30000, progress_bar=True)
print('학습 완료')

## 3. 학습 후 성능 비교

학습한 에이전트의 평균 보상을 다시 측정해 학습 전과 비교합니다. CartPole 최대 점수는 500입니다.

In [ ]:
mean_after, std_after = evaluate_policy(model, env, n_eval_episodes=20)
print(f'학습 후 평균 보상: {mean_after:.1f} +/- {std_after:.1f}')

plt.figure(figsize=(5, 4))
plt.bar(['학습 전', '학습 후'], [mean_before, mean_after],
        yerr=[std_before, std_after], color=['#bbbbbb', '#4c72b0'], capsize=5)
plt.ylabel('평균 보상'); plt.title('PPO 학습 전후 비교 (CartPole)')
plt.ylim(0, 520)
for i, v in enumerate([mean_before, mean_after]):
    plt.text(i, v + 10, f'{v:.0f}', ha='center')
plt.tight_layout(); plt.show()

env.close()

### 정리

- SB3의 PPO로 CartPole을 학습해, 무작위 수준(20~30점)에서 최대 점수(최대 500점)에 근접하도록 향상시켰습니다.
- `evaluate_policy`로 학습 전후 성능을 정량 비교했습니다.
- 환경(`CartPole-v1`)만 바꾸면 같은 코드로 다른 제어 문제도 학습할 수 있습니다.